# scverse AgentCore — Conversational Research Notebook

Talk to the single-cell genomics agent in **plain English**. You don't write scanpy or scvi-tools code — you ask, the agent runs live compute on AWS, and plots render inline.

This notebook is a thin **chat loop** over the deployed **Amazon Bedrock AgentCore Runtime** agent. Every message goes:

```
notebook  →  AgentCore Runtime (Strands agent)  →  AgentCore Gateway (MCP)  →  Lambda / Fargate / Step Functions  →  S3
```

As the agent works, the tools it calls (for example `list_datasets`, `run_scrna_qc`) print live so you can see what is happening while compute runs.

**Prerequisites:** local AWS credentials for the account where the stack is deployed (`aws sts get-caller-identity` should work), and the agent deployed to AgentCore Runtime. See `frontend/README.md` for setup.

## Setup

In [ ]:
import json, re, uuid, boto3
from IPython.display import display, Image, Markdown

REGION = "us-west-2"

# Discover the deployed agent ARN from SSM (written by deploy.py).
# Fall back to pasting it directly if you prefer.
ssm = boto3.client("ssm", region_name=REGION)
try:
    AGENT_ARN = ssm.get_parameter(Name="/agents/scverse_research_agent_arn")["Parameter"]["Value"]
except Exception:
    AGENT_ARN = "arn:aws:bedrock-agentcore:us-west-2:ACCOUNT:runtime/scverse_research_agent-XXXX"  # <-- paste yours

agentcore = boto3.client("bedrock-agentcore", region_name=REGION)
SESSION_ID = f"notebook-{uuid.uuid4().hex}{uuid.uuid4().hex}"[:40]

# Client-side conversation memory. The deployed agent is stateless per call by
# default (AgentCore Memory is optional and off unless AGENTCORE_MEMORY_ID is
# set), so the notebook threads context by sending a short transcript each turn.
_HISTORY = []

def reset_chat():
    """Forget the conversation so far (start a fresh context)."""
    _HISTORY.clear()
    print("Conversation reset.")

print("Agent:", AGENT_ARN)
print("Identity:", boto3.client("sts", region_name=REGION).get_caller_identity()["Arn"])

In [ ]:
_IMG_MD = re.compile(r"!\[([^\]]*)\]\((https?://[^)\s]+)\)")
_TOOL_MARK = "🔧 Using tool:"

# LLM output is untrusted. Strip raw HTML/scripts and neutralize dangerous
# link schemes before rendering as Markdown, so a malicious agent response
# can't inject JavaScript in the notebook (defense-in-depth XSS guard).
_SCRIPT_STYLE = re.compile(r"<(script|style)\b[^>]*>.*?</\1>", re.IGNORECASE | re.DOTALL)
_HTML_TAG = re.compile(r"</?[a-zA-Z][^>]*>")  # real HTML tags only; leaves math like "genes < 2500 and counts > 500"
_BAD_LINK = re.compile(r"\]\(\s*(?:javascript|data|vbscript):", re.IGNORECASE)

def _safe_md(text):
    """Return Markdown-safe text: no raw HTML tags or script/JS URIs."""
    text = _SCRIPT_STYLE.sub("", text)
    text = _HTML_TAG.sub("", text)
    text = _BAD_LINK.sub("](#", text)
    return text

def _tool_label(name):
    # Gateway tools arrive prefixed, e.g. "ScverseLambdaTarget___list_datasets".
    return name.split("___")[-1].strip()

def chat(prompt, session_id=SESSION_ID, remember=True):
    """Send a natural-language message to the scverse agent and render the reply.

    - Prints each tool the agent calls live, so you can see progress while
      live compute runs (QC can take 20-60s).
    - Threads prior turns client-side so follow-up questions keep context.
    - Text renders as Markdown; any ![](url) plots render inline as images.
    """
    if not prompt.strip():
        print("Type a question inside the quotes, then re-run this cell.")
        return

    # Fold prior conversation into the prompt (client-side memory).
    if _HISTORY:
        transcript = "\n\n".join(f"{role.upper()}: {text}" for role, text in _HISTORY[-6:])
        full_prompt = ("Continue this conversation. Use the prior turns for context.\n\n"
                       f"{transcript}\n\nUSER: {prompt}")
    else:
        full_prompt = prompt

    print("⏳ Sending to the agent — live compute can take 20-60s...", flush=True)
    resp = agentcore.invoke_agent_runtime(
        agentRuntimeArn=AGENT_ARN,
        payload=json.dumps({"prompt": full_prompt}),
        runtimeSessionId=session_id,
        qualifier="DEFAULT",
    )

    # AgentCore returns an SSE event stream; iterate it so tool calls surface live.
    body = resp.get("response")
    if hasattr(body, "iter_lines"):
        line_iter = body.iter_lines()
    else:
        line_iter = (body.read().decode("utf-8", "replace") if body else "").split("\n")

    chunks, shown_tools = [], set()
    for raw in line_iter:
        line = raw.decode("utf-8", "replace") if isinstance(raw, (bytes, bytearray)) else raw
        line = line.strip()
        if not line.startswith("data: "):
            continue
        content = line[6:]
        if content.startswith('"') and content.endswith('"'):
            content = json.loads(content)  # handles escape sequences
        chunks.append(content)
        if _TOOL_MARK in content:
            tool = _tool_label(content.split(_TOOL_MARK, 1)[1])
            if tool and tool not in shown_tools:
                shown_tools.add(tool)
                print(f"   🔧 {tool}", flush=True)

    text = "".join(chunks).strip() or "(empty response)"
    # Clean answer for display + history (drop the inline tool markers we showed live).
    clean = re.sub(r"\n*" + re.escape(_TOOL_MARK) + r"[^\n]*\n*", "\n", text).strip()
    print("✅ Done.\n", flush=True)

    # Render text and images so plots show inline.
    last = 0
    for m in _IMG_MD.finditer(clean):
        before = clean[last:m.start()].strip()
        if before:
            display(Markdown(_safe_md(before)))
        display(Image(url=m.group(2)))
        last = m.end()
    tail = clean[last:].strip()
    if tail:
        display(Markdown(_safe_md(tail)))

    # Update client-side memory (store text only; collapse plot URLs to stay compact).
    if remember:
        _HISTORY.append(("user", prompt))
        _HISTORY.append(("assistant", _IMG_MD.sub("[plot]", clean)))

print("chat() ready — ask the agent anything below.")

## 1. Discover your data

In [ ]:
chat("What datasets are available?")

## 2. Run QC in plain English (plots render inline)

In [ ]:
chat("Run QC on the PBMC dataset with a 5% mitochondrial threshold and show me the plots")

## 3. Follow up conversationally

The notebook threads the conversation client-side (it sends a short transcript with each turn), so you can iterate the way you'd talk to a colleague. Run this right after the QC cell above.

> The deployed agent is stateless per call by default. For server-side memory across sessions, enable AgentCore Memory (set `AGENTCORE_MEMORY_ID` on the Runtime); the agent already supports it.

In [ ]:
chat("Why did those 302 cells get filtered out? Was 5% too strict?")

## 4. Ask for methodology guidance

In [ ]:
chat("Which scvi-tools model should I use for CITE-seq data, and why?")

## 5. Your turn

Edit the prompt below and run. Try: submitting an async pipeline, checking a job status, or asking the agent to explain a result. Call `reset_chat()` any time to start a fresh conversation.

In [ ]:
chat("")  # <-- type your question inside the quotes